In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
df = pd.read_csv("youtube_ad_revenue_dataset.csv")
df.head()

,video_id,date,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,ad_revenue_usd
0,vid_3092,2024-09-24 10:50:40.993199,9936.0,1221.0,320.0,26497.214184,2.862137,228086.0,Entertainment,TV,IN,203.178237
1,vid_3459,2024-09-22 10:50:40.993199,10017.0,642.0,346.0,15209.747445,23.738069,736015.0,Gaming,Tablet,CA,140.880508
2,vid_4784,2024-11-21 10:50:40.993199,10097.0,1979.0,187.0,57332.658498,26.200634,240534.0,Education,TV,CA,360.134008
3,vid_4078,2025-01-28 10:50:40.993199,10034.0,1191.0,242.0,31334.517771,11.770340,434482.0,Entertainment,Mobile,UK,224.638261
4,vid_3522,2025-04-28 10:50:40.993199,9889.0,1858.0,477.0,15665.666434,6.635854,42030.0,Education,Mobile,CA,165.514388


In [3]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
df.fillna(method='ffill', inplace=True)

/tmp/ipykernel_1686/1168271076.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [4]:
df['engagement_rate'] = (df['likes'] + df['comments']) / df['views']

In [5]:
le = LabelEncoder()

df['category'] = le.fit_transform(df['category'])
df['device'] = le.fit_transform(df['device'])
df['country'] = le.fit_transform(df['country'])

In [6]:
X = df.drop(['ad_revenue_usd', 'video_id', 'date'], axis=1)
y = df['ad_revenue_usd']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Initialize and fit StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

models = {
    "Linear": LinearRegression(),
    "RandomForest": RandomForestRegressor(),
    "DecisionTree": DecisionTreeRegressor(),
    "SVR": SVR()
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)

    print(f"\n{name}")
    print("R2:", r2_score(y_test, pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print("MAE:", mean_absolute_error(y_test, pred))


Linear
R2: 0.9152909715561237
RMSE: 18.425064913765834
MAE: 6.475813512599104

RandomForest
R2: 0.9029962966962223
RMSE: 19.716886369572062
MAE: 8.04645787647752

DecisionTree
R2: 0.7872293309863571
RMSE: 29.201156767333234
MAE: 10.055106274454092

SVR
R2: 0.8910423068659022
RMSE: 20.896479543058817
MAE: 10.599613852510263


In [9]:
import pickle
import os

best_model = RandomForestRegressor()
best_model.fit(X_train_scaled, y_train)

os.makedirs('model/', exist_ok=True)
pickle.dump(best_model, open("model/revenue_model.pkl", "wb"))
pickle.dump(scaler, open("model/scaler.pkl", "wb")) # Save the fitted scaler

In [10]:
!pip install streamlit
import streamlit as st
import pickle
import numpy as np

model = pickle.load(open("model/revenue_model.pkl", "rb"))

st.title("💰 YouTube Revenue Predictor")

views = st.number_input("Views")
likes = st.number_input("Likes")
comments = st.number_input("Comments")
watch_time = st.number_input("Watch Time (minutes)")
video_length = st.number_input("Video Length")
subscribers = st.number_input("Subscribers")

category = st.selectbox("Category", [0,1,2,3])
device = st.selectbox("Device", [0,1,2])
country = st.selectbox("Country", [0,1,2])

engagement_rate = (likes + comments) / views if views != 0 else 0

if st.button("Predict Revenue"):
    data = np.array([[views, likes, comments, watch_time, video_length,
                      subscribers, category, device, country, engagement_rate]])

    prediction = model.predict(data)

    st.success(f"Estimated Revenue: ${prediction[0]:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.5 MB/s eta 0:00:00


2026-05-08 05:55:47.509 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.715 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-08 05:55:47.716 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.717 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.720 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.721 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.724 Thread 'MainThread': mi

In [11]:
import pickle
import streamlit as st
import numpy as np

# ---------------- LOAD MODEL & SCALER ---------------- #

with open("model/revenue_model.pkl", "rb") as file:
    model = pickle.load(file)

with open("model/scaler.pkl", "rb") as file:
    scaler = pickle.load(file)


# ---------------- SIDEBAR ---------------- #

st.sidebar.title('📊 Navigation')
selection = st.sidebar.radio('Go to', ['Home', 'Predict'])


# ---------------- HOME PAGE ---------------- #

if selection == 'Home':
    st.title('💰 Content Monetization Modeler')

    st.image("https://images.unsplash.com/photo-1611162617474-5b21e879e113", use_column_width=True)

    st.write("""
    This application predicts **YouTube video ad revenue** using machine learning.

    It analyzes key performance metrics such as:
    - Views
    - Likes & Comments
    - Watch Time
    - Subscribers

    🎯 Goal:
    Help creators and businesses estimate revenue and optimize content strategy.
    """)

    st.markdown("### 🔍 Features")
    st.write("""
    ✔ Revenue Prediction
    ✔ Engagement Analysis
    ✔ Real-time Input
    ✔ ML-powered Insights
    """)


# ---------------- PREDICTION PAGE ---------------- #

elif selection == 'Predict':

    st.title('📈 Predict YouTube Revenue')

    col1, col2 = st.columns(2)

    # -------- LEFT COLUMN -------- #
    with col1:
        views = st.number_input('Views', min_value=0.0)
        likes = st.number_input('Likes', min_value=0.0)
        comments = st.number_input('Comments', min_value=0.0)
        watch_time = st.number_input('Watch Time (minutes)', min_value=0.0)
        video_length = st.number_input('Video Length (minutes)', min_value=0.0)

    # -------- RIGHT COLUMN -------- #
    with col2:
        subscribers = st.number_input('Subscribers', min_value=0.0)

        category = st.selectbox(
            'Category',
            ['Entertainment', 'Education', 'Gaming', 'Music']
        )

        device = st.selectbox(
            'Device',
            ['Mobile', 'Desktop', 'Tablet']
        )

        country = st.selectbox(
            'Country',
            ['India', 'USA', 'UK', 'Canada']
        )

    # -------- ENCODING -------- #

    category_map = {'Entertainment': 0, 'Education': 1, 'Gaming': 2, 'Music': 3}
    device_map = {'Mobile': 0, 'Desktop': 1, 'Tablet': 2}
    country_map = {'India': 0, 'USA': 1, 'UK': 2, 'Canada': 3}

    category_val = category_map[category]
    device_val = device_map[device]
    country_val = country_map[country]

    # -------- FEATURE ENGINEERING -------- #

    engagement_rate = (likes + comments) / views if views != 0 else 0

    # -------- PREDICTION -------- #

    if st.button('💰 Predict Revenue'):

        data = np.array([[views, likes, comments, watch_time, video_length,
                          subscribers, category_val, device_val,
                          country_val, engagement_rate]])

        scaled_data = scaler.transform(data)

        prediction = model.predict(scaled_data)

        st.success(f"💵 Estimated Revenue: ${prediction[0]:.2f}")

        # -------- INSIGHTS -------- #
        st.subheader("📊 Insights")

        if engagement_rate > 0.1:
            st.write("🔥 High engagement → Better monetization")

        if views > 100000:
            st.write("🚀 High views → Strong revenue potential")

        if watch_time > video_length:
            st.write("⏱️ Good retention → Positive signal")

2026-05-08 05:55:47.976 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.978 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.979 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.981 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.982 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-08 05:55:47.989 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [12]:
%%writefile app.py
import pickle
import streamlit as st
import numpy as np

# ---------------- LOAD MODEL & SCALER ---------------- #

with open("model/revenue_model.pkl", "rb") as file:
    model = pickle.load(file)

with open("model/scaler.pkl", "rb") as file:
    scaler = pickle.load(file)


# ---------------- SIDEBAR ---------------- #

st.sidebar.title('📊 Navigation')
selection = st.sidebar.radio('Go to', ['Home', 'Predict'])


# ---------------- HOME PAGE ---------------- #

if selection == 'Home':
    st.title('💰 Content Monetization Modeler')

    st.image("https://images.unsplash.com/photo-1611162617474-5b21e879e113", use_column_width=True)

    st.write("""
    This application predicts **YouTube video ad revenue** using machine learning.

    It analyzes key performance metrics such as:
    - Views
    - Likes & Comments
    - Watch Time
    - Subscribers

    🎯 Goal:
    Help creators and businesses estimate revenue and optimize content strategy.
    """)

    st.markdown("### 🔍 Features")
    st.write("""
    ✔ Revenue Prediction
    ✔ Engagement Analysis
    ✔ Real-time Input
    ✔ ML-powered Insights
    """)


# ---------------- PREDICTION PAGE ---------------- #

elif selection == 'Predict':

    st.title('📈 Predict YouTube Revenue')

    col1, col2 = st.columns(2)

    # -------- LEFT COLUMN -------- #
    with col1:
        views = st.number_input('Views', min_value=0.0)
        likes = st.number_input('Likes', min_value=0.0)
        comments = st.number_input('Comments', min_value=0.0)
        watch_time = st.number_input('Watch Time (minutes)', min_value=0.0)
        video_length = st.number_input('Video Length (minutes)', min_value=0.0)

    # -------- RIGHT COLUMN -------- #
    with col2:
        subscribers = st.number_input('Subscribers', min_value=0.0)

        category = st.selectbox(
            'Category',
            ['Entertainment', 'Education', 'Gaming', 'Music']
        )

        device = st.selectbox(
            'Device',
            ['Mobile', 'Desktop', 'Tablet']
        )

        country = st.selectbox(
            'Country',
            ['India', 'USA', 'UK', 'Canada']
        )

    # -------- ENCODING -------- #

    category_map = {'Entertainment': 0, 'Education': 1, 'Gaming': 2, 'Music': 3}
    device_map = {'Mobile': 0, 'Desktop': 1, 'Tablet': 2}
    country_map = {'India': 0, 'USA': 1, 'UK': 2, 'Canada': 3}

    category_val = category_map[category]
    device_val = device_map[device]
    country_val = country_map[country]

    # -------- FEATURE ENGINEERING -------- #

    engagement_rate = (likes + comments) / views if views != 0 else 0

    # -------- PREDICTION -------- #

    if st.button('💰 Predict Revenue'):

        data = np.array([[views, likes, comments, watch_time, video_length,
                          subscribers, category_val, device_val,
                          country_val, engagement_rate]])

        scaled_data = scaler.transform(data)

        prediction = model.predict(scaled_data)

        st.success(f"💵 Estimated Revenue: ${prediction[0]:.2f}")

        # -------- INSIGHTS -------- #
        st.subheader("📊 Insights")

        if engagement_rate > 0.1:
            st.write("🔥 High engagement → Better monetization")

        if views > 100000:
            st.write("🚀 High views → Strong revenue potential")

        if watch_time > video_length:
            st.write("⏱️ Good retention → Positive signal")

Writing app.py


In [2]:
!pip install streamlit -q

In [3]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 118198 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [8]:
!streamlit run app.py &>/content/logs.txt &

In [9]:
!cloudflared tunnel --url http://localhost:8501 &>/dev/null &

In [10]:
# Kill any existing cloudflared processes to ensure a clean start
!killall cloudflared 2>/dev/null || true

# Start cloudflared tunnel, capturing its output to a temporary log file
# This ensures we can extract the generated public URL.
!cloudflared tunnel --url http://localhost:8501 > cloudflared_temp_url.log 2>&1 &

import time
import re

# Give cloudflared a few seconds to start up and write the URL to the log file
time.sleep(5)

# Read the log file and extract the URL
url_found = None
try:
    with open("cloudflared_temp_url.log", "r") as f:
        log_content = f.read()
        # Regex to find the public URL, which typically starts with https:// and ends with .trycloudflare.com
        match = re.search(r"https://\S+\.trycloudflare\.com", log_content)
        if match:
            url_found = match.group(0)
except FileNotFoundError:
    print("Cloudflared log file not found.")

if url_found:
    print(f"\nYour Streamlit app is accessible at: {url_found}")
else:
    print("\nFailed to obtain Cloudflare Tunnel URL. Please check 'cloudflared_temp_url.log' for details.")



Your Streamlit app is accessible at: https://observation-rare-diana-need.trycloudflare.com
